# Production ONI SPT Batch Pipeline
Run complete ONI SPT datasets from explicit YAML configurations. There is no pilot stage: `Run All` performs preflight, then resumes or starts the production batch.

## 1. Batch controls
Processing parameters live in the YAML files, not in notebook cells.

In [ ]:
from pathlib import Path

batch_config = Path('configs/production/batch.yaml')
resume_existing = True
force_replace = False  # Keep False unless intentionally replacing a reviewed run.
execute_batch = True

print('Batch:', batch_config.resolve())
print('Resume:', resume_existing, '| Force:', force_replace, '| Execute:', execute_batch)

## 2. Preflight
Confirm accepted files, frames, rejected inputs, storage, and preliminary ETA before processing.

In [ ]:
from IPython.display import display
from run_ONI_SPT_batch import preflight_batch

batch, prepared, preflight = preflight_batch(batch_config)
display(preflight)
print('Accepted files:', int(preflight.accepted_files.sum()))
print('Accepted frames:', int(preflight.accepted_frames.sum()))
print(f"Estimated outputs: {preflight.estimated_output_GiB.sum():.1f} GiB")
print(f"Free space: {preflight.attrs['free_GiB']:.1f} GiB")

## 3. Run or resume the complete batch
Progress and ETA are printed after every file and stored inside each result folder.

In [ ]:
from run_ONI_SPT_batch import run_batch

if execute_batch:
    run_batch(batch_config, resume=resume_existing, force=force_replace)
else:
    print('Preflight only: execute_batch is False.')

## 4. Results and QC index

In [ ]:
import pandas as pd
from spt_shared import output_dirs

rows = []
for item in prepared:
    cfg = item['cfg']; dirs = output_dirs(cfg['output_dir'])
    rows.append({
        'dataset': cfg['dataset'],
        'manifest': dirs['metadata'] / 'input_manifest.csv',
        'progress': dirs['metadata'] / 'batch_progress.json',
        'report': dirs['reports'] / 'diffusion_comparison_report.pdf',
        'detection_QC_videos': len(list((dirs['qc'] / 'detection_videos').glob('*.mp4'))),
        'trajectory_QC_videos': len(list((dirs['qc'] / 'trajectory_videos').glob('*.mp4'))),
    })
display(pd.DataFrame(rows))